# PyEvoc Step-by-Step Pipeline Tutorial

This notebook guides a new user through the current PyEvoc package structure.

It follows the actual module architecture:

```text
pyevoc.data
pyevoc.preprocessing
pyevoc.features
pyevoc.analysis
pyevoc.visualisation
```

The workflow is intentionally explicit. Each step can be run, inspected, and modified.

## 0. How to use PyEvoc locally

If PyEvoc is not installed from PyPI, the recommended local usage is:

```bash
git clone https://github.com/text-lab/pyevoc.git
cd pyevoc
pip install -e .
```

The `-e` option installs the package in editable mode, so changes to the local package files are immediately visible to the notebook.

Alternatively, install directly from GitHub:

```bash
pip install git+https://github.com/text-lab/pyevoc.git
```

## 1. Import core libraries and inspect PyEvoc

This cell verifies that Python can import the local PyEvoc package.

To simplify the procedure, ensure the dataset, the anchor list, and this notebook file are in the same folder.

In [ ]:
import os
import pandas as pd

import pyevoc

print("PyEvoc imported from:")
print(pyevoc.__file__)

## 2. Load and standardise the dataset

PyEvoc expects a standard corpus schema with four required columns:

```text
user_id | doc_id | time | text
```

Your original dataset can have any column names. You only need to map your original column names to the PyEvoc standard names.

For example, if your file contains:

```text
account_ID | post_id | publication_time | content
```

you map them like this:

```python
column_map={
    "account_id":       "user_id",
    "post_id":          "doc_id",
    "publication_time": "time",
    "content":          "text",
}
```

Any additional columns present in the source file are preserved automatically after the four required ones.

In [ ]:
from pyevoc.data.dataset import DatasetConfig, load_dataset, standardise_dataset, corpus_summary

# -----------------------------------------------------------------------
# Change this path to your input file.
# Supported formats: .csv, .tsv, .xlsx, .parquet, .json
# -----------------------------------------------------------------------
INPUT_FILE = "AGI.xlsx"

# Adapt the left-hand side to match your dataset's column names.
# The right-hand side must use PyEvoc standard names.
config = DatasetConfig(
    column_map={
        "account_ID":   "user_id",
        "tweet_ID":    "doc_id",
        "tweet_pub_time":   "time",
        "tweet_text":   "text",
    },
    start_date="2022-10-01",
    end_date="2025-05-01",
    timezone="UTC",
    drop_missing_text=True,
)

corpus = load_dataset(INPUT_FILE, config=config)

print(corpus.shape)

## 3. Language Filtering

PyEvoc implements a two-stage language identification workflow based on fastText and Lingua. 

It works with English, Italian, French, German, Spanish, Portuguese.

The primary detection stage relies on the official fastText language-identification model (`lid.176.bin` or `lid.176.ftz`), which provides efficient and accurate language classification for large textual corpora.

If the fastText model is not already available locally, it can be downloaded automatically using:

```python
from pyevoc.preprocessing.language_filtering import download_fasttext_lid_model

model_path = download_fasttext_lid_model(compact=False)
```

For documents with uncertain fastText classifications, PyEvoc can optionally apply a secondary validation step based on the Lingua language detector. To enable this functionality, install the additional dependency:

```python
pip install lingua-language-detector
```

For large corpora, language filtering may take time.

In [ ]:
from pyevoc.preprocessing.language_filtering import (
    LanguageFilterConfig,
    download_fasttext_lid_model,
    filter_by_language,
)

model_path = download_fasttext_lid_model(compact=False)

lang_config = LanguageFilterConfig(
    text_col="text",
    target_language="en",
    min_probability=0.90,
    ambiguous_min_probability=0.50,
    ambiguous_max_probability=0.90,
    use_lingua_fallback=True,
    lingua_min_confidence=0.80,
    keep_probability=True,
    keep_diagnostics=False,
    verbose=False,
)

subcorpus_lin = filter_by_language(
    corpus,
    config=lang_config,
    model_path=model_path,
)

print(subcorpus_lin.shape)

## 4. Thematic filtering with an external anchor list

The thematic filter extracts a domain-specific subcorpus.

The anchor file should be a plain text file with one anchor term or phrase per line.

In [ ]:
from pyevoc.preprocessing.thematic_filtering import (
    ThematicFilterConfig,
    build_thematic_subset,
    load_anchor_terms
)

ANCHOR_FILE = "anchor.txt"  # change this path

theme_config = ThematicFilterConfig(
    text_col="text",
    min_anchor_hits=1,
    similarity_threshold=0.10,
    min_expansion_hits=2,
    max_features=50000,
    lowercase=True,
    ngram_range=(1, 2)
)

subcorpus_the, theme_metadata = build_thematic_subset(
    subcorpus_lin,
    anchor_file=ANCHOR_FILE,
    config=theme_config
)

print(subcorpus_the.shape)

## 5. Subcorpus statistics

This step gives basic descriptive information about the selected corpus.

In [ ]:
from pyevoc.preprocessing.corpus_statistics import corpus_statistics

stats = corpus_statistics(
    subcorpus_lin,     #if only language filtering is applied
#   subcorpus_the,     #if language and thematic filtering is applied
    text_col="text",
)

stats

## 6. Basic text cleaning

The cleaned text is stored in a new column called `clean_text`.

In [ ]:
from pyevoc.preprocessing.cleaning import CleaningConfig, clean_corpus

clean_config = CleaningConfig(
    lowercase=True,
    remove_urls=True,
    url_placeholder="URL",
    expand_contractions=True,
    space_emojis=True,
    reduce_elongated_vowels=True,
    space_punctuation=True,
    show_progress=True,
)

corpus_clean = clean_corpus(
    subcorpus_lin,     #if only language filtering is applied
#   subcorpus_the,     #if language and thematic filtering is applied
    text_col="text",
    output_col="clean_text",
    config=clean_config,
)

## 7. Linguistic annotation with Stanza

This step performs tokenisation, lemmatisation, POS tagging, and named-entity recognition.

The output is a token-level dataframe.

If Stanza models are not installed, you may need to run:

```python
import stanza
stanza.download("en")
```

In [ ]:
from pyevoc.preprocessing import (
    AnnotationConfig,
    annotate_with_stanza,
    annotation_diagnostics,
    dependency_diagnostics,
)

ann_config = AnnotationConfig(
    text_col="clean_text",
    doc_id_col="doc_id",
    user_col="user_id",
    time_col="time",
    language="en",
    processors="tokenize,pos,lemma,depparse",
    use_gpu=True,
    batch_size=128,
    save_parquet=True,
    output_dir="annotated_outputs",
    output_name="corpus_stanza_anno",
)

tokens_raw = annotate_with_stanza(
    corpus_clean,
    config=ann_config,
)

In [ ]:
#RECOVERY FROM ANNOTATION BACKUP

import pandas as pd
import os

OUTPUT_DIR = "annotated_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

tokens_raw = pd.read_parquet(
    os.path.join(OUTPUT_DIR, "corpus_stanza_anno.parquet")
)

## 8. Emoji assignment

Emoji tokens are reassigned to the `EMOJI` UPOS category.

In [ ]:
from pyevoc.features.emoji_assignment import assign_emoji_upos

tokens, emoji_diag, emoji_top = assign_emoji_upos(
    tokens_raw,
    token_col="token",
    upos_col="upos",
    lemma_col="lemma",
    return_diagnostics=True,
)

## 9. Structural foregrounding and positional salience

This step computes the indicators needed to reconstruct the AOE-like salience component.

In [ ]:
from pyevoc.features.foregrounding import (
    ForegroundingConfig,
    ForegroundingWeights,
    add_salience_indicators,
    foregrounding_diagnostics,
    metadata_coverage,
)

metadata_coverage(tokens)

fg_config = ForegroundingConfig(
    doc_col="doc_id",
    user_col="user_id",
    time_col="time",
    token_col="token",
    sentence_col="sentence_id",
    position_col="position",
    weights=ForegroundingWeights(
        opening_sentence=0.35,
        emphasis=0.25,
        list_or_quote=0.20,
        intensification=0.20,
    ),
)

tokens = add_salience_indicators(tokens, config=fg_config)

foregrounding_diagnostics(tokens)

## 10. Unigram cleaning and selection

This step defines the lexical units retained for representational analysis.

In [ ]:
from pyevoc.features.unigram_selection import select_unigrams

tokens_selected, unigram_diag, unigram_params = select_unigrams(
    tokens,
    keep_upos={"NOUN", "ADJ", "EMOJI"},
    token_col="token",
    lemma_col="lemma",
    upos_col="upos",
    doc_col="doc_id",
    user_col="user_id",
    min_docs_per_term=3,
    min_users_per_term=3,
    return_diagnostics=True,
)

unigram_diag

## 11. Compute AFE/AOE term-level indices

This step computes diffusion, salience, and rank-like AOE indicators.

In [ ]:
from pyevoc.features.term_indices import (
    SalienceConfig,
    compute_term_statistics,
)

term_stats, pos_thresholds, quadrant_summary = compute_term_statistics(
    tokens_selected,
    term_col="term",
    upos_col="upos",
    user_col="user_id",
    doc_col="doc_id",
    time_col="time",
    r_pos_col="r_pos",
    r_str_col="r_str",
    config=SalienceConfig(
        alpha=0.50,
        max_rank_value=5.0,
        use_users_for_frequency=True,
        focal_upos={"NOUN", "ADJ", "EMOJI"},
    ),
)

pos_thresholds

## 12. Concreteness labelling

This step adds concreteness labels when a lexicon is available.

In [ ]:
from pyevoc.features.concreteness import (
    load_concreteness_lexicon,
    add_concreteness_labels,
)

lexicon = load_concreteness_lexicon()

term_stats, concreteness_coverage = add_concreteness_labels(
    term_stats,
    lexicon=lexicon,
)

concreteness_coverage

## 13. Emoji labelling

This step adds descriptions for emoji terms and falls back to Unicode names when needed.

In [ ]:
from pyevoc.features.emoji_labelling import (
    load_emoji_lookup,
    add_emoji_descriptions,
)

emoji_lookup = load_emoji_lookup()

term_stats, emoji_coverage, emoji_methods, missing_emoji = add_emoji_descriptions(
    term_stats,
    lookup=emoji_lookup,
    min_similarity=0.80,
)

emoji_coverage

## 14. EVOC thresholds and quadrants

Thresholds are computed separately by POS category.

In [ ]:
from pyevoc.analysis.quadrants import assign_evoc_quadrants

evoc_quadrants, quadrant_counts, pos_thresholds_round = assign_evoc_quadrants(
    term_stats,
    generate_html=True,
    html_output_dir="evoc_outputs",
    html_top_n=5,
)

evoc_quadrants.attrs["html_outputs"]

## 15. Collocations and Named Entities

This extracts collocational structures (and optionally named entities) from the selected token table.

In [ ]:
from pyevoc.analysis.collocations_entities import extract_collocations_and_entities

colloc_results = extract_collocations_and_entities(
    tokens_df=tokens_raw,
    include_collocations=True,
    include_entities=False,
    min_freq=2,
    min_docs=2,
    min_users=2,
    write_html=True,
)

collocations_df = colloc_results["collocations_df"]

collocations_df.shape

#results = extract_collocations_and_entities(
#    tokens_df=tokens,
#    include_collocations=True,
#    include_entities=True,
#    entity_n=2,
#    resolve_overlap=True,
#    prefer_overlap="evidence",
#    write_html=True,
#)
#
#collocations_df = results["collocations_df"]
#entities_df = results["entities_df"]
#overlap_log_df = results["overlap_log_df"]

## 16. Temporal stability

Temporal stability analysis is performed by partitioning the corpus into time periods and recomputing the EVOC framework within each period. Depending on the analytical objective, periods may be defined using custom temporal boundaries, equal-duration intervals, or approximately equal-interval intervals.

In [ ]:
from pyevoc.analysis.temporal_stability import run_temporal_stability_analysis

temporal_results = run_temporal_stability_analysis(
    df=tokens_selected,
    evoc_quadrants_df=evoc_quadrants,
    pos_thresholds_round_df=pos_thresholds_round,
    n_periods=4,
    time_period_mode="equal_days",
    show_tables=False,
)

#temporal_results = run_temporal_stability_analysis(
#    df=tokens_selected,
#    evoc_quadrants_df=evoc_quadrants,
#    pos_thresholds_round_df=pos_thresholds_round,
#    time_period_mode="custom",
#    custom_time_cuts=["2022-01-01", "2024-01-01"],
#    period_labels=["2018-2021", "2022-2023", "2024-2026"],
#    show_table=False,
#)

## 17. Visualisations

The package currently provides plotting functions for EVOC maps, emoji maps, semantic-tree edges, and Sankey diagrams.

In [ ]:
from pyevoc.visualisation import (
    build_evoc_target_plot,
    build_evoc_collocation_tree_for_upos,
    build_emoji_evoc_plot,
    build_temporal_sankey_ordered,
)

OUTPUT_DIR = "evoc_outputs"

# 1. EVOC target maps
plot_nouns = build_evoc_target_plot(
    evoc_quadrants,
    "NOUN",
    pos_thresholds_round,
    output_dir=OUTPUT_DIR,
    node_size_mode="fixed",
)

plot_adjectives = build_evoc_target_plot(
    evoc_quadrants,
    "ADJ",
    pos_thresholds_round,
    output_dir=OUTPUT_DIR,
    node_size_mode="fixed",
)

# 2. EVOC semantic trees
tree_nouns = build_evoc_collocation_tree_for_upos(
    evoc_quadrants_df=evoc_quadrants,
    collocations_df=collocations_df,
    tokens_df=tokens_raw,
    upos="NOUN",
    output_dir=OUTPUT_DIR,
    top_roots_per_quadrant=5,
    max_leaves_per_root=3,
    min_leaf_freq=5,
)

tree_adjectives = build_evoc_collocation_tree_for_upos(
    evoc_quadrants_df=evoc_quadrants,
    collocations_df=collocations_df,
    tokens_df=tokens_raw,
    upos="ADJ",
    output_dir=OUTPUT_DIR,
    top_roots_per_quadrant=5,
    max_leaves_per_root=3,
    min_leaf_freq=5,
)

# 3. EVOC emoji map
fig_emoji, emoji_evoc_df, emoji_evoc_html = build_emoji_evoc_plot(
    html_file=f"{OUTPUT_DIR}/evoc_quadrants_emoji_compact.html",
    output_dir=OUTPUT_DIR,
    output_png=None,
)

# 4. EVOC temporal sankey
sankey_results = build_temporal_sankey_ordered(
    period_quadrants=temporal_results["period_quadrants"],
    output_dir=OUTPUT_DIR,
    output_html="evoc_temporal_sankey.html",
    output_png=None,
    arrangement="snap",
    show_zero_nodes=False,
    width=1000,
    height=500,
)

# End of tutorial

At this stage you have produced:

- a standardised corpus;
- a language-filtered corpus;
- a thematic subcorpus;
- a cleaned and annotated token table;
- term-level AFE/AOE indicators;
- EVOC quadrants;
- collocations;
- HTML reports;
- visual outputs.

For detailed theoretical explanations, consult:

```text
docs/methodology.md
```